In [ ]:
# Import required libraries
import requests
import pandas as pd
import json

In [ ]:
# Set API endpoint and fallback account ID
API_BASE_URL = "http://localhost:3000/api/v1/lakehouse/network-analysis/transaction/"
FALLBACK_ACCOUNT_ID = "dbtrAcct_24a03dafa2c14f6da6bfc195d57c6d21"

In [ ]:
def fetch_transaction_network(account_id, api_base_url=API_BASE_URL):
    url = f"{api_base_url}{account_id}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json(), response.status_code
    except requests.RequestException as e:
        print(f"Error fetching data: {e}")
        return None, None

In [ ]:
# Fetch transaction data for accountId from URL query param if present, else fallback
import os
from urllib.parse import parse_qs, urlparse
from IPython.display import display, HTML
import sys

def get_account_id_from_url():
    if 'VOILA_BASE_URL' in os.environ:
        if len(sys.argv) > 0 and '?' in sys.argv[0]:
            query = urlparse(sys.argv[0]).query
            params = parse_qs(query)
            return params.get('accountId', [FALLBACK_ACCOUNT_ID])[0]
    # Fallback: use fallback account id
    return FALLBACK_ACCOUNT_ID

account_id = get_account_id_from_url()
transaction_data, status_code = fetch_transaction_network(account_id)
# Only display the chart, no print or data dump


In [ ]:
import networkx as nx
import plotly.graph_objects as go

# Build the network graph from the API data
def build_network_graph(data):
    G = nx.DiGraph()
    # Add center node
    center = data['centerAccount']
    G.add_node(center['accountId'], label=center['accountHolder'], type='center', hasAlert=False)
    # Add connected accounts
    for acc in data.get('connectedAccounts', []):
        G.add_node(acc['accountId'], label=acc['accountHolder'], type='connected', hasAlert=acc['hasAlert'])
    # Add edges
    for edge in data.get('edges', []):
        G.add_edge(edge['source'], edge['target'], type=edge['type'])
    return G

if transaction_data:
    G = build_network_graph(transaction_data)
else:
    G = None

In [ ]:
import plotly.graph_objects as go
import networkx as nx
from IPython.display import display, HTML
import json

display(
    HTML(
        """
        <style>
            body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; margin: 0; padding: 0; background: #F9FAFB; }
            .legend-title { font-size: 18px; font-weight: 600; color: #111827; margin-bottom: 16px; }
            .legend-item { display: flex; align-items: center; gap: 12px; margin-bottom: 12px; font-size: 14px; color: #374151; }
            .legend-marker { width: 20px; height: 20px; border-radius: 50%; border: 2px solid #000; }
            .legend-marker.center { background: #F59E0B; border-width: 2px; }
            .legend-marker.normal { background: #6366F1; border-width: 2px; }
            .legend-marker.alert { background: #EF4444; border-width: 2px; }
            .legend-edge { width: 40px; height: 2px; background: #9CA3AF; border-top: 2px dashed #9CA3AF; }
            .section-title { font-size: 18px; font-weight: 600; color: #111827; margin-bottom: 24px; }
            .field { margin-bottom: 20px; }
            .field-label { font-size: 12px; font-weight: 500; color: #6B7280; text-transform: uppercase; margin-bottom: 4px; }
            .field-value { font-size: 14px; color: #111827; word-break: break-all; }
            .network-summary { margin-top: 32px; padding-top: 20px; border-top: 1px solid #E5E7EB; }
            .network-summary-title { font-size: 16px; font-weight: 600; color: #111827; margin-bottom: 16px; }
            .network-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; font-size: 14px; color: #374151; }
        </style>
        """
    )
)


def _layout_center_middle(data):
    import math

    center = data['centerAccount']
    conn = data.get('connectedAccounts', [])
    pos = {center['accountId']: (0.5, 0.5)}
    n = len(conn)
    r = 0.35
    for i, acc in enumerate(conn):
        th = 2 * math.pi * i / n - math.pi / 2 if n else 0
        pos[acc['accountId']] = (0.5 + r * math.cos(th), 0.5 + r * math.sin(th))
    return pos


def _no_data_html():
    req = _last_request or {}
    err = _last_error or {}
    url = req.get('url', '')
    params = req.get('params', {})

    details = ""
    if err.get('type') == 'http':
        details = (
            f"<div style='margin-top:8px; color:#991B1B;'>HTTP {err.get('status','')} {err.get('reason','')}</div>"
            f"<pre style='margin-top:8px; padding:10px; background:#111827; color:#F9FAFB; border-radius:8px; overflow:auto; max-height:200px;'>{(err.get('body') or '')}</pre>"
        )
    elif err.get('type') == 'exception':
        details = f"<div style='margin-top:8px; color:#991B1B;'>Request failed: {err.get('message','')}</div>"

    return f"""
        <div style='padding: 12px; border: 1px solid #E5E7EB; border-radius: 8px; background: #F9FAFB;'>
          <div style='color:#6B7280;'>No network data to plot.</div>
          <div style='margin-top:8px; color:#111827; font-weight:600;'>Request</div>
          <div style='margin-top:4px; color:#374151; word-break: break-all;'><span style='color:#6B7280;'>URL:</span> {url}</div>
          <pre style='margin-top:8px; padding:10px; background:#FFFFFF; border:1px solid #E5E7EB; border-radius:8px; overflow:auto;'>{json.dumps(params, indent=2)}</pre>
          {details}
        </div>
    """


def plot_network(G, data):
    if G is None or len(G.nodes) == 0 or not data:
        display(HTML(_no_data_html()))
        return

    pos = _layout_center_middle(data)

    edge_x, edge_y = [], []
    for edge in G.edges(data=True):
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    node_x, node_y, node_text, node_color, node_border, customdata = [], [], [], [], [], []
    for node, nd in G.nodes(data=True):
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(f"{node}<br>{nd.get('label','')}")
        if nd['type'] == 'center':
            node_color.append('#F59E0B')
            node_border.append(10)
        elif nd['hasAlert']:
            node_color.append('#EF4444')
            node_border.append(6)
        else:
            node_color.append('#6366F1')
            node_border.append(4)
        customdata.append({
            'id': node,
            'label': nd.get('label', ''),
            'type': nd['type'],
            'hasAlert': nd.get('hasAlert', False),
        })

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        line=dict(width=2, color='#9CA3AF', dash='dash'),
        hoverinfo='none',
        mode='lines',
        showlegend=False,
    )
    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        marker=dict(
            size=[28] * len(node_x),
            color=node_color,
            line=dict(width=node_border, color='black'),
        ),
        text=node_text,
        textposition='bottom center',
        hoverinfo='text',
        customdata=customdata,
        showlegend=False,
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            showlegend=False,
            hovermode='closest',
            margin=dict(b=20, l=5, r=5, t=40),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        ),
    )
    fig.update_layout(
        title='Transaction Network — center account in middle',
        height=500,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )

    center_node = {
        'id': data['centerAccount']['accountId'],
        'label': data['centerAccount']['accountHolder'],
        'type': 'center',
        'hasAlert': False,
    }

    details_html = f"""
        <div id="details-container" style="display: flex; width: 100%; gap: 16px; margin-top: 16px;">
          <div style="width: 200px; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
            <div class="legend-title">Legend</div>
            <div class="legend-item"><div class="legend-marker center"></div>Center Account</div>
            <div class="legend-item"><div class="legend-marker normal"></div>Normal Account</div>
            <div class="legend-item"><div class="legend-marker alert"></div>Alert Triggered</div>
            <div class="legend-item"><div class="legend-edge"></div>Transaction Flow</div>
          </div>
          <div style="flex: 1; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
            <div class="section-title">Account Details</div>
            <div class="field">
              <div class="field-label">Account ID</div>
              <div class="field-value">{center_node['id']}</div>
            </div>
            <div class="field">
              <div class="field-label">Account Holder</div>
              <div class="field-value">{center_node['label']}</div>
            </div>
            <div class="field">
              <div class="field-label">Type</div>
              <div class="field-value">{'Center' if center_node['type'] == 'center' else 'Connected'}</div>
            </div>
            <div class="field">
              <div class="field-label">Alert Status</div>
              <div class="field-value">{'Alert Triggered' if center_node['hasAlert'] else 'Normal'}</div>
            </div>
            <div class="network-summary">
              <div class="network-summary-title">Network Summary</div>
              <div class="network-grid">
                <div><strong>Connected Accounts:</strong> 0</div>
                <div><strong>Outbound Connections:</strong> 0</div>
                <div><strong>Inbound Connections:</strong> 0</div>
                <div><strong>Accounts with Alerts:</strong> 0</div>
              </div>
            </div>
          </div>
        </div>
    """

    display(HTML(details_html))
    fig.show()
    display(
        HTML(
            f"""
            <script>
            window.networkData = {json.dumps(data)};
            function updateAccountDetails(nodeData) {{
                const container = document.getElementById('details-container');
                if (container && window.networkData) {{
                    const data = window.networkData;
                    const nodeId = nodeData.id;
                    const label = nodeData.label;
                    const nodeType = nodeData.type;
                    const hasAlert = nodeData.hasAlert;

                    let inbound = 0, outbound = 0, connected = 0, alerts = 0;
                    if (data && data.edges) {{
                        for (const e of data.edges) {{
                            if (e.target === nodeId) inbound++;
                            if (e.source === nodeId) outbound++;
                        }}
                    }}
                    if (data && data.connectedAccounts) {{
                        connected = data.connectedAccounts.filter(a => a.accountId === nodeId).length;
                        alerts = data.connectedAccounts.filter(a => a.accountId === nodeId && a.hasAlert).length;
                    }}

                    const rowHtml = `
                        <div style="width: 200px; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
                          <div class="legend-title">Legend</div>
                          <div class="legend-item"><div class="legend-marker center"></div>Center Account</div>
                          <div class="legend-item"><div class="legend-marker normal"></div>Normal Account</div>
                          <div class="legend-item"><div class="legend-marker alert"></div>Alert Triggered</div>
                          <div class="legend-item"><div class="legend-edge"></div>Transaction Flow</div>
                        </div>
                        <div style="flex: 1; background: #FFFFFF; border: 1px solid #E5E7EB; border-radius: 8px; padding: 24px; box-shadow: 0 1px 2px 0 rgba(0,0,0,0.05);">
                          <div class="section-title">Account Details</div>
                          <div class="field">
                            <div class="field-label">Account ID</div>
                            <div class="field-value">${{nodeId || ''}}</div>
                          </div>
                          <div class="field">
                            <div class="field-label">Account Holder</div>
                            <div class="field-value">${{label || ''}}</div>
                          </div>
                          <div class="field">
                            <div class="field-label">Type</div>
                            <div class="field-value">${{nodeType === 'center' ? 'Center' : 'Connected'}}</div>
                          </div>
                          <div class="field">
                            <div class="field-label">Alert Status</div>
                            <div class="field-value">${{hasAlert ? 'Alert Triggered' : 'Normal'}}</div>
                          </div>
                          <div class="network-summary">
                            <div class="network-summary-title">Network Summary</div>
                            <div class="network-grid">
                              <div><strong>Connected Accounts:</strong> ${{connected}}</div>
                              <div><strong>Outbound Connections:</strong> ${{outbound}}</div>
                              <div><strong>Inbound Connections:</strong> ${{inbound}}</div>
                              <div><strong>Accounts with Alerts:</strong> ${{alerts}}</div>
                            </div>
                          </div>
                        </div>
                    `;
                    container.innerHTML = rowHtml;
                }}
            }}
            document.addEventListener('DOMContentLoaded', function() {{
                setTimeout(function() {{
                    const gd = document.getElementsByClassName('plotly-graph-div')[0];
                    if (gd) {{
                        gd.on('plotly_click', function(data) {{
                            if (data.points && data.points.length) {{
                                const point = data.points[0];
                                if (point.customdata) {{
                                    updateAccountDetails(point.customdata);
                                }}
                            }}
                        }});
                    }}
                }}, 1000);
            }});
            </script>
            """
        )
    )


plot_network(G, transaction_data)